# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*


### Two paper findings + my methodology questions

I reviewed FlyRank's research paper, *The State of AI-Driven SEO in Numbers* (March 2026). I selected two findings and considered what methodology questions I would ask before applying the findings to decisions.

#### Finding 1: The Anatomy of Growing Content

The paper reports that growing content averaged approximately 3,200 words and 184 days of age, while declining content averaged approximately 2,300 words and 230 days of age.

The paper describes this as an observational comparison, not proof that longer content or younger age causes growth.

**My methodology question:**

How exactly was the growth or decline label defined? Since the paper defines trend direction using the change in impressions between the latest 30 days and the previous 30 days, were pages with insufficient data, flat trends, or newly published content handled separately?

Understanding how the label was created would help me judge what the comparison actually measures and avoid interpreting an observed relationship as causation.

#### Finding 2: The Content Performance Curve

The paper reports that content health peaked around 61–90 days, declined in the 271–365 day group, and showed a higher average health score in the 365+ day group. The paper explains that the older-content recovery was concentrated among pages that had been refreshed.

It also clarifies that this does not mean age naturally reverses performance decline.

**My methodology question:**

Were refreshed and unrefreshed pages compared within similar age groups and with similar starting visibility, content quality, and demand? Could the older refreshed pages already have been stronger candidates before they were updated?

A matched comparison or time-aware before-and-after analysis could help separate the observed association with freshness from other factors that may influence performance.

#### My takeaway

Both findings provide useful directional evidence, but they should be interpreted within their observational design and data limitations.

For my own model, I will apply the same standard: inspect how the target label is created, prevent information leakage, use an honest validation split, and describe results as measured decision-support signals rather than causal conclusions.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

### My model under an honest split (before/after)

In Week 5, I used a single 80/20 holdout grouped by `client_id`. This prevents pages from the same client appearing in both training and test data.

For this audit, I compare that single grouped holdout with 5-fold `GroupKFold` validation. In GroupKFold, each client is held out in one fold, giving a broader view of performance across clients.

I keep the model, features, and evaluation metrics consistent. The comparison measures validation sensitivity and stability; it is not evidence that the model itself improved.

I report Average Precision and Precision@20. These are ranking metrics for identifying declining pages near the top of a review list.

In [ ]:
# Week 6: reproduce Week 5 grouped holdout and add 5-fold GroupKFold.
# This cell is self-contained and uses the repository's official feature lists.

import sys
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Find or clone the repository
repo_candidates = [
    Path.cwd(),
    Path("/content/Flyrank-ML-Internship"),
    Path("/content"),
]

REPO_DIR = next(
    (
        folder for folder in repo_candidates
        if (folder / "scripts" / "01_prepare_features.py").exists()
    ),
    None
)

if REPO_DIR is None:
    REPO_DIR = Path("/content/Flyrank-ML-Internship")
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/rashmita-21/Flyrank-ML-Internship.git",
            str(REPO_DIR)
        ],
        check=True
    )

RAW_PATH = REPO_DIR / "data" / "raw" / "content_refresh_anonymized.csv"
if not RAW_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {RAW_PATH}. Check that the repository contains "
        "data/raw/content_refresh_anonymized.csv."
    )

# Run the repository's official preparation script
subprocess.run(
    [sys.executable, str(REPO_DIR / "scripts" / "01_prepare_features.py")],
    cwd=REPO_DIR,
    check=True
)

sys.path.insert(0, str(REPO_DIR / "scripts"))
from ml_utils import MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES

df = pd.read_csv(
    REPO_DIR / "data" / "processed" / "refresh_feature_vector.csv"
)

# Match the Week 5 target and official feature lists
model_df = df.dropna(
    subset=["is_declining_label", "client_id"]
).copy()

model_df["is_declining_label"] = (
    model_df["is_declining_label"].astype(int)
)

feature_columns = list(dict.fromkeys(
    [
        col for col in MODEL_NUMERIC_FEATURES
        if col in model_df.columns
    ] +
    [
        col for col in MODEL_CATEGORICAL_FEATURES
        if col in model_df.columns
    ]
))

# Explicitly exclude IDs and direct target-construction fields
prohibited = {
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "is_declining_label",
}
feature_columns = [
    col for col in feature_columns if col not in prohibited
]

X = model_df[feature_columns].copy()
y = model_df["is_declining_label"].copy()
groups = model_df["client_id"].astype(str)

numeric_features = [
    col for col in MODEL_NUMERIC_FEATURES
    if col in feature_columns
]
categorical_features = [
    col for col in MODEL_CATEGORICAL_FEATURES
    if col in feature_columns
]

# Same preprocessing and Logistic Regression approach as Week 5
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])

def build_model():
    return Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42,
        )),
    ])

def precision_at_k(y_true, scores, k):
    y_array = np.asarray(y_true)
    score_array = np.asarray(scores)
    k = min(k, len(y_array))
    top_indices = np.argsort(-score_array)[:k]
    return float(y_array[top_indices].mean())

def calculate_metrics(y_true, scores):
    return {
        "Average Precision": average_precision_score(y_true, scores),
        "Precision@20": precision_at_k(y_true, scores, 20),
        "Precision@50": precision_at_k(y_true, scores, 50),
    }

# BEFORE: reproduce the Week 5 single 80/20 client-grouped holdout
holdout_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42,
)

train_idx, test_idx = next(
    holdout_splitter.split(X, y, groups=groups)
)

train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])

assert train_clients.isdisjoint(test_clients), (
    "Client overlap found in Week 5 holdout."
)

holdout_model = build_model()
holdout_model.fit(X.iloc[train_idx], y.iloc[train_idx])
holdout_scores = holdout_model.predict_proba(X.iloc[test_idx])[:, 1]

holdout_metrics = calculate_metrics(
    y.iloc[test_idx],
    holdout_scores,
)

# AFTER: 5-fold GroupKFold out-of-fold predictions
n_clients = groups.nunique()
if n_clients < 5:
    raise ValueError(
        f"GroupKFold requires at least 5 clients; found {n_clients}."
    )

group_kfold = GroupKFold(n_splits=5)

oof_scores = cross_val_predict(
    build_model(),
    X,
    y,
    groups=groups,
    cv=group_kfold,
    method="predict_proba",
    n_jobs=-1,
)[:, 1]

oof_metrics = calculate_metrics(y, oof_scores)

# Display the before/after validation comparison
validation_comparison = pd.DataFrame([
    {"Validation": "Week 5: one client holdout", **holdout_metrics},
    {"Validation": "Week 6: 5-fold GroupKFold OOF", **oof_metrics},
])

print("Rows:", len(model_df))
print("Unique clients:", n_clients)
print("Features:", len(feature_columns))
print("Week 5 holdout train/test client overlap:", len(train_clients & test_clients))
print("Week 5 holdout rows:", len(test_idx))
print("Week 6 out-of-fold rows:", len(oof_scores))
print("Overall declining-label rate:", round(y.mean(), 4))

display(validation_comparison.round(4))

# Keep these values available for the later notebook sections
week5_holdout_metrics = holdout_metrics
week6_groupkfold_metrics = oof_metrics
week6_oof_scores = oof_scores

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.